# MAESTRO MIDI song-string databases and matching

This notebook uses the expanded MAESTRO dataset prepared in `maestro_train` and `maestro_songs`. The training directory contains the longest recording for each normalized canonical song, and the grouped song directory contains every MIDI recording under its canonical-song directory.

The notebook extracts full-song highest-note base strings, builds one lookup database per string-processing approach, matches every grouped query recording against those databases, and reports compression, accuracy, and runtime results.


In [35]:
from collections import Counter
from pathlib import Path
import base64
import pickle
import zlib
import mido
import pandas as pd
import os

# Dataset options: "mini", "maestro", or "maestro_multi".
# "mini" includes just the three songs with the most recordings
# "maestro_multi" includes all songs with more than one recording
# "maestro" includes all songs
DATASET_NAME = "maestro"

# Set to None to run all approaches or to a specific approach ID to run just that approach
SELECTED_APPROACH_ID = None

# Cap parallel MIDI parsing to avoid oversubscribing the CPU on the mini dataset.
PARALLEL_WORKERS = os.cpu_count()

# Persist extracted MIDI strings so repeated notebook runs avoid reparsing unchanged files.
USE_EXTRACTION_CACHE = True
EXTRACTION_CACHE_DIR = Path(".cache")

DATASET_CONFIGS = {
    "mini": {
        "train_dir": Path("mini_train"),
        "songs_dir": Path("mini_songs"),
        "require_multiple_recordings": False,
    },
    "maestro": {
        "train_dir": Path("maestro_train"),
        "songs_dir": Path("maestro_songs"),
        "require_multiple_recordings": False,
    },
    "maestro_multi": {
        "train_dir": Path("maestro_train"),
        "songs_dir": Path("maestro_songs"),
        "require_multiple_recordings": True,
    },
}

if DATASET_NAME not in DATASET_CONFIGS:
    raise ValueError(f"Unknown DATASET_NAME: {DATASET_NAME!r}. Choose from {sorted(DATASET_CONFIGS)}.")

DATASET_CONFIG = DATASET_CONFIGS[DATASET_NAME]
TRAIN_DIR = DATASET_CONFIG["train_dir"]
SONGS_DIR = DATASET_CONFIG["songs_dir"]
REQUIRE_MULTIPLE_RECORDINGS = DATASET_CONFIG["require_multiple_recordings"]
EXTRACTION_CACHE_PATH = EXTRACTION_CACHE_DIR / "midi_extractions.pkl"

ONSET_GROUP_SECONDS = 0.08
PIANO_LOWEST_MIDI_NOTE = 21
PIANO_HIGHEST_MIDI_NOTE = 108
PRINTABLE_ASCII = [chr(code) for code in range(33, 127)]

## MIDI extraction helpers

These functions parse MIDI files, group near-simultaneous note onsets, select notes from each onset group, and encode selected MIDI notes as separator-free ASCII strings.


In [36]:
def extract_note_onsets(path):
    """Extract note-on events from a MIDI file with absolute times in seconds.

    Args:
        path: Path to a MIDI file.

    Returns:
        A list of dictionaries, each containing the onset time in seconds, MIDI note number, note name, and velocity.
    """
    midi = mido.MidiFile(path)
    tempo = 500_000
    seconds = 0.0
    note_onsets = []

    for message in mido.merge_tracks(midi.tracks):
        seconds += mido.tick2second(message.time, midi.ticks_per_beat, tempo)

        if message.type == "set_tempo":
            tempo = message.tempo
        elif message.type == "note_on" and message.velocity > 0:
            note_onsets.append(
                {
                    "time": seconds,
                    "note": message.note,
                    "velocity": message.velocity,
                }
            )

    return note_onsets


In [37]:
def group_note_onsets(note_onsets, onset_group_seconds=ONSET_GROUP_SECONDS):
    """Group nearby note-on events into onset time steps.

    Args:
        note_onsets: Note-on dictionaries returned by `extract_note_onsets`.
        onset_group_seconds: Maximum time span from the first onset in a group for
            notes to be treated as the same time step.

    Returns:
        A list of onset groups, where each group is a list of note-on dictionaries.
    """
    if not note_onsets:
        return []

    start_time = note_onsets[0]["time"]
    groups = []
    current_group = []
    group_start = None

    for note in note_onsets:
        aligned_time = note["time"] - start_time
        if not current_group or aligned_time - group_start <= onset_group_seconds:
            if not current_group:
                group_start = aligned_time
            current_group.append(note)
        else:
            groups.append(current_group)
            current_group = [note]
            group_start = aligned_time

    if current_group:
        groups.append(current_group)
    return groups

In [38]:
def select_notes_from_groups(groups, selection="highest", k=1):
    """Select notes from already-grouped MIDI onsets.

    Args:
        groups: List of onset groups returned by `group_note_onsets`.
        selection: Selection strategy: ``"highest"``, ``"lowest"``, or ``"top_k"``.
        k: Maximum number of notes per onset group when ``selection`` is ``"top_k"``.

    Returns:
        A flat list of MIDI note numbers selected from all onset groups.
    """
    selected_notes = []

    for group in groups:
        if selection == "highest":
            selected_notes.append(max(group, key=lambda onset: onset["note"])["note"])
        elif selection == "lowest":
            selected_notes.append(min(group, key=lambda onset: onset["note"])["note"])
        elif selection == "top_k":
            top_notes = sorted(group, key=lambda onset: onset["note"], reverse=True)[:k]
            selected_notes.extend(note["note"] for note in sorted(top_notes, key=lambda onset: onset["note"]))
        else:
            raise ValueError("selection must be 'highest', 'lowest', or 'top_k'")

    return selected_notes

In [39]:
def build_piano_note_character_map(lowest_note=PIANO_LOWEST_MIDI_NOTE, highest_note=PIANO_HIGHEST_MIDI_NOTE):
    """Build a deterministic one-character ASCII encoding for piano MIDI notes.

    Args:
        lowest_note: Lowest supported MIDI note number. Defaults to A0, the lowest piano key.
        highest_note: Highest supported MIDI note number. Defaults to C8, the highest piano key.

    Returns:
        A dictionary mapping each supported MIDI note number to a unique printable ASCII character.
    """
    supported_notes = list(range(lowest_note, highest_note + 1))
    if len(supported_notes) > len(PRINTABLE_ASCII):
        raise ValueError("The requested note range needs more printable ASCII characters than are available")
    return {note: PRINTABLE_ASCII[index] for index, note in enumerate(supported_notes)}


In [40]:
def encode_notes_as_ascii(notes, note_character_map):
    """Encode a note list as a base string string.

    Args:
        notes: Sequence of MIDI note numbers.
        note_character_map: Mapping from MIDI note number to unique ASCII character.

    Returns:
        A string containing one ASCII character per note and no separators.
    """
    unsupported_notes = sorted({note for note in notes if note not in note_character_map})
    if unsupported_notes:
        raise ValueError(f"Unsupported notes outside the ASCII map: {unsupported_notes}")
    return "".join(note_character_map[note] for note in notes)


In [41]:
def extraction_settings_for_name(extraction):
    """Return note-selection settings for a named extraction strategy.

    Args:
        extraction: Extraction name: ``"highest"``, ``"lowest"``, ``"top2"``, or ``"top3"``.

    Returns:
        A dictionary with ``selection`` and ``k`` entries for the extraction strategy.
    """
    extraction_settings = {
        "highest": {"selection": "highest", "k": 1},
        "lowest": {"selection": "lowest", "k": 1},
        "top2": {"selection": "top_k", "k": 2},
        "top3": {"selection": "top_k", "k": 3},
    }
    if extraction not in extraction_settings:
        raise ValueError(f"Unknown extraction type: {extraction!r}. Choose from {sorted(extraction_settings)}.")
    return extraction_settings[extraction]

In [42]:
def midi_file_to_note_strings_by_extraction(path, note_character_map, extractions):
    """Convert one MIDI file to multiple extraction strings with one MIDI parse.

    Args:
        path: Path to a MIDI file.
        note_character_map: Mapping from MIDI note number to unique ASCII character.
        extractions: Iterable of extraction names to produce.

    Returns:
        A dictionary mapping extraction name to separator-free ASCII note string.
    """
    note_onsets = extract_note_onsets(path)
    groups = group_note_onsets(note_onsets)
    strings_by_extraction = {}

    for extraction in extractions:
        settings = extraction_settings_for_name(extraction)
        notes = select_notes_from_groups(
            groups,
            selection=settings["selection"],
            k=settings["k"],
        )
        strings_by_extraction[extraction] = encode_notes_as_ascii(notes, note_character_map)

    return strings_by_extraction

In [43]:
def extraction_cache_key(path):
    """Build a stable cache key for one MIDI file version.

    Args:
        path: Path to a MIDI file.

    Returns:
        A tuple containing the resolved path, file size, and modification time.
    """
    path = Path(path)
    stat = path.stat()
    return (str(path.resolve()), stat.st_size, stat.st_mtime_ns)

In [44]:
def load_extraction_cache(cache_path, enabled=True):
    """Load the persistent MIDI extraction cache if available.

    Args:
        cache_path: Pickle file path for the cache.
        enabled: Whether persistent caching should be used.

    Returns:
        A mutable dictionary mapping file-version keys to cached extraction strings.
    """
    if not enabled or not cache_path.exists():
        return {}
    with cache_path.open("rb") as cache_file:
        return pickle.load(cache_file)

In [45]:
def save_extraction_cache(cache_path, cache, enabled=True):
    """Write the persistent MIDI extraction cache to disk.

    Args:
        cache_path: Pickle file path for the cache.
        cache: Cache dictionary to persist.
        enabled: Whether persistent caching should be used.

    Returns:
        None.
    """
    if not enabled:
        return None
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("wb") as cache_file:
        pickle.dump(cache, cache_file)
    return None

In [46]:
def cached_midi_file_to_note_strings(path, note_character_map, extractions, cache):
    """Return extraction strings from cache, parsing the MIDI file only on misses.

    Args:
        path: Path to a MIDI file.
        note_character_map: Mapping from MIDI note number to unique ASCII character.
        extractions: Iterable of extraction names to produce.
        cache: Extraction cache loaded by `load_extraction_cache`.

    Returns:
        A tuple with requested extraction strings, elapsed retrieval/parse time in
        milliseconds, the file cache key, and the complete updated cache entry.
    """
    import time

    requested_extractions = tuple(sorted(extractions))
    key = extraction_cache_key(path)
    cached_strings = dict(cache.get(key, {}))
    missing_extractions = [extraction for extraction in requested_extractions if extraction not in cached_strings]

    start = time.perf_counter()
    if missing_extractions:
        cached_strings.update(
            midi_file_to_note_strings_by_extraction(
                path,
                note_character_map,
                missing_extractions,
            )
        )
    elapsed_ms = (time.perf_counter() - start) * 1000

    return (
        {extraction: cached_strings[extraction] for extraction in requested_extractions},
        elapsed_ms,
        key,
        cached_strings,
    )

## String-processing helpers

These functions transform extracted note strings by filtering repeats, collapsing motifs, summarizing fixed-size blocks, abstracting pitch, or applying generic compression.


In [47]:
def repeated_run_signature(note_string, min_run_length=2):
    """Keep only repeated-character runs and collapse each kept run to one character.

    Args:
        note_string: Separator-free ASCII note string.
        min_run_length: Minimum adjacent run length required for a character to be kept.

    Returns:
        A string containing one character for each adjacent run whose length is at least `min_run_length`.
    """
    if min_run_length <= 0:
        raise ValueError("min_run_length must be positive")
    if not note_string:
        return ""

    signature = []
    current_character = note_string[0]
    count = 1

    for character in note_string[1:]:
        if character == current_character:
            count += 1
        else:
            if count >= min_run_length:
                signature.append(current_character)
            current_character = character
            count = 1

    if count >= min_run_length:
        signature.append(current_character)
    return "".join(signature)


In [48]:
def collapse_repeated_chunks(note_string, chunk_size):
    """Collapse adjacent repeated chunks of a fixed length to one copy.

    Args:
        note_string: Separator-free ASCII note string.
        chunk_size: Number of characters in each chunk to compare.

    Returns:
        A string where immediately repeated chunks of `chunk_size` are represented once.
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    output = []
    previous_chunk = None
    index = 0

    while index < len(note_string):
        chunk = note_string[index : index + chunk_size]
        if chunk != previous_chunk:
            output.append(chunk)
            previous_chunk = chunk
        index += chunk_size

    return "".join(output)


In [49]:
def collapse_repeated_motifs(note_string, min_chunk_size=2, max_chunk_size=8):
    """Collapse adjacent repeated motifs using several chunk sizes.

    Args:
        note_string: Separator-free ASCII note string.
        min_chunk_size: Smallest motif length to try.
        max_chunk_size: Largest motif length to try.

    Returns:
        The shortest string found after applying fixed-size repeated-chunk collapse across the requested motif sizes.
    """
    candidates = [collapse_repeated_chunks(note_string, chunk_size) for chunk_size in range(min_chunk_size, max_chunk_size + 1)]
    return min(candidates, key=len) if candidates else note_string


In [50]:
def modal_block_signature(note_string, block_size=8):
    """Represent each fixed-size block by its most common character.

    Args:
        note_string: Separator-free ASCII note string.
        block_size: Number of characters per block.

    Returns:
        A string containing one modal character per block.
    """
    if block_size <= 0:
        raise ValueError("block_size must be positive")

    signature = []
    for index in range(0, len(note_string), block_size):
        block = note_string[index : index + block_size]
        most_common_character = Counter(block).most_common(1)[0][0]
        signature.append(most_common_character)
    return "".join(signature)


In [51]:
def first_character_blocks(note_string, block_size=8):
    """Represent each fixed-size block by its first character.

    Args:
        note_string: Separator-free ASCII note string.
        block_size: Number of characters per block.

    Returns:
        A string containing the first character from each fixed-size block.
    """
    if block_size <= 0:
        raise ValueError("block_size must be positive")
    return "".join(note_string[index] for index in range(0, len(note_string), block_size))


In [52]:
def pitch_band_signature(note_string, band_count=8):
    """Map note characters into coarse pitch bands and collapse adjacent repeated bands.

    Args:
        note_string: Separator-free ASCII note string whose character order follows ascending pitch.
        band_count: Number of coarse pitch bands to use.

    Returns:
        A compact string over `band_count` ASCII symbols representing coarse pitch movement.
    """
    if band_count <= 0:
        raise ValueError("band_count must be positive")
    if not note_string:
        return ""

    note_span = PIANO_HIGHEST_MIDI_NOTE - PIANO_LOWEST_MIDI_NOTE + 1
    banded = []
    for character in note_string:
        note_index = ord(character) - ord(PRINTABLE_ASCII[0])
        band_index = min(band_count - 1, note_index * band_count // note_span)
        banded.append(PRINTABLE_ASCII[band_index])
    return repeated_run_signature("".join(banded), min_run_length=1)


In [53]:
def pitch_class_signature(note_string):
    """Map note characters to pitch-class characters and collapse adjacent repeats.

    Args:
        note_string: Separator-free ASCII note string whose character order follows ascending pitch.

    Returns:
        A string over 12 ASCII symbols representing pitch classes rather than exact octaves.
    """
    pitch_classes = []
    for character in note_string:
        pitch_class = (ord(character) - ord(PRINTABLE_ASCII[0]) + PIANO_LOWEST_MIDI_NOTE) % 12
        pitch_classes.append(PRINTABLE_ASCII[pitch_class])
    return repeated_run_signature("".join(pitch_classes), min_run_length=1)


In [54]:
def contour_signature(note_string):
    """Convert a note string to a compressed up/down/same contour signature.

    Args:
        note_string: Separator-free ASCII note string whose character order follows ascending pitch.

    Returns:
        A string over `U`, `D`, and `S` with consecutive repeated contour symbols collapsed.
    """
    if len(note_string) < 2:
        return note_string

    contour = []
    for previous, current in zip(note_string, note_string[1:]):
        if current > previous:
            contour.append("U")
        elif current < previous:
            contour.append("D")
        else:
            contour.append("S")

    return repeated_run_signature("".join(contour), min_run_length=1)


In [55]:
def zlib_base85_encode(note_string):
    """Compress a note string with zlib and encode the bytes as ASCII.

    Args:
        note_string: Separator-free ASCII note string.

    Returns:
        An ASCII string containing the base85 representation of the zlib-compressed bytes.
    """
    compressed = zlib.compress(note_string.encode("ascii"), level=9)
    return base64.b85encode(compressed).decode("ascii")


## Matching helpers

These functions build exact/Jaccard or vectorized n-gram matchers and use them to predict the closest training song for a processed query string.


In [56]:
def string_ngrams(value, n=3):
    """Convert a string into a set of character n-grams for fast similarity scoring.

    Args:
        value: String to convert into n-grams.
        n: Character n-gram size.

    Returns:
        A set of character n-gram strings, or individual characters when the input is shorter than `n`.
    """
    if len(value) < n:
        return set(value)
    return {value[index : index + n] for index in range(len(value) - n + 1)}


In [57]:
def build_match_index(database, n=3):
    """Precompute exact and inverted n-gram indexes for a string-to-song database.

    Args:
        database: Dictionary mapping processed training strings to song names.
        n: Character n-gram size for approximate matching.

    Returns:
        A dictionary containing exact lookup data, per-song n-gram profiles, and an inverted n-gram index for faster approximate matching.
    """
    profiles = []
    inverted_index = {}

    for database_key, song_name in database.items():
        ngrams = string_ngrams(database_key, n=n)
        profile_index = len(profiles)
        profiles.append(
            {
                "database_key": database_key,
                "song_name": song_name,
                "ngrams": ngrams,
                "ngram_count": len(ngrams),
            }
        )
        for ngram in ngrams:
            inverted_index.setdefault(ngram, []).append(profile_index)

    return {
        "exact": dict(database),
        "profiles": profiles,
        "inverted_index": inverted_index,
        "n": n,
    }


In [58]:
def match_song_with_index(processed_query_string, match_index):
    """Predict the closest training song using a precomputed inverted match index.

    Args:
        processed_query_string: Query string after applying the same transform used for the database.
        match_index: Precomputed match index returned by `build_match_index`.

    Returns:
        A dictionary containing the predicted song name, similarity score, and whether the match was exact.
    """
    exact_database = match_index["exact"]
    if processed_query_string in exact_database:
        return {"predicted_song": exact_database[processed_query_string], "score": 1.0, "exact": True}

    profiles = match_index["profiles"]
    if not profiles:
        return {"predicted_song": None, "score": 0.0, "exact": False}

    query_ngrams = string_ngrams(processed_query_string, n=match_index["n"])
    overlap_counts = {}
    for ngram in query_ngrams:
        for profile_index in match_index["inverted_index"].get(ngram, []):
            overlap_counts[profile_index] = overlap_counts.get(profile_index, 0) + 1

    if not overlap_counts:
        return {"predicted_song": profiles[0]["song_name"], "score": 0.0, "exact": False}

    query_ngram_count = len(query_ngrams)
    best_profile_index = None
    best_score = -1.0

    for profile_index, overlap_count in overlap_counts.items():
        profile = profiles[profile_index]
        union_count = query_ngram_count + profile["ngram_count"] - overlap_count
        score = overlap_count / union_count if union_count else 1.0
        if score > best_score:
            best_score = score
            best_profile_index = profile_index

    return {"predicted_song": profiles[best_profile_index]["song_name"], "score": best_score, "exact": False}


In [59]:
def build_vectorized_matcher(train_strings_by_song, matcher_type="tfidf", ngram_range=(1, 1)):
    """Build a vectorized character n-gram matcher for a training string database.

    Uses scikit-learn TF-IDF or binary-count vectorization so every training string
    is projected into a shared feature space.  Querying reduces to a single
    sparse matrix-vector product instead of iterating over every database entry.

    Args:
        train_strings_by_song: Mapping from song name to processed training string.
        matcher_type: ``'tfidf'`` for TF-IDF cosine similarity or ``'binary'`` for
            L2-normalised binary character n-gram cosine similarity.
        ngram_range: Tuple ``(min_n, max_n)`` specifying the character n-gram range
            passed to the underlying scikit-learn vectorizer.

    Returns:
        A dictionary with keys ``'vectorizer'``, ``'matrix'``, ``'song_names'``, and
        ``'matcher_type'`` for use with `match_with_vectorized_matcher`.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
    from sklearn.preprocessing import normalize

    song_names = list(train_strings_by_song.keys())
    strings = list(train_strings_by_song.values())

    if matcher_type == "tfidf":
        vectorizer = TfidfVectorizer(analyzer="char", ngram_range=ngram_range)
        matrix = vectorizer.fit_transform(strings)
    elif matcher_type == "binary":
        vectorizer = CountVectorizer(analyzer="char", binary=True, ngram_range=ngram_range)
        matrix = normalize(vectorizer.fit_transform(strings), norm="l2")
    else:
        raise ValueError(f"Unknown matcher_type: {matcher_type!r}. Choose 'tfidf' or 'binary'.")

    return {
        "vectorizer": vectorizer,
        "matrix": matrix,
        "song_names": song_names,
        "matcher_type": matcher_type,
    }


In [60]:
def match_with_vectorized_matcher(query_string, matcher):
    """Predict the closest training song using a prebuilt vectorized character n-gram matcher.

    Args:
        query_string: Processed query string, using the same transform as the training database.
        matcher: Prebuilt matcher dictionary returned by `build_vectorized_matcher`.

    Returns:
        A dictionary with keys ``'predicted_song'``, ``'score'``, and ``'exact'``.
    """
    from sklearn.metrics.pairwise import linear_kernel
    from sklearn.preprocessing import normalize

    query_vec = matcher["vectorizer"].transform([query_string])
    if matcher["matcher_type"] == "binary":
        query_vec = normalize(query_vec, norm="l2")
    scores = linear_kernel(query_vec, matcher["matrix"]).flatten()
    best_idx = int(scores.argmax())
    return {
        "predicted_song": matcher["song_names"][best_idx],
        "score": float(scores[best_idx]),
        "exact": False,
    }


In [61]:
def build_approach_matcher(spec, train_strings):
    """Build one processed-string database and matcher for an approach.

    Args:
        spec: Approach specification dictionary from `ACTIVE_APPROACH_SPECS`.
        train_strings: Mapping from training song name to extracted note string for
            the approach's selected extraction method.

    Returns:
        A dictionary containing the approach spec, matcher, database build time,
        original string length, processed string length, and shortening metrics.
    """
    import time

    transform = spec["transform"]
    start_build = time.perf_counter()
    processed_train_strings = {song_name: transform(base_string) for song_name, base_string in train_strings.items()}
    original_key_length = sum(len(value) for value in train_strings.values())
    processed_key_length = sum(len(value) for value in processed_train_strings.values())

    if spec["matcher_type"] == "indexed_jaccard":
        database = {processed_string: song_name for song_name, processed_string in processed_train_strings.items()}
        matcher = build_match_index(database, n=spec["ngram_range"][0])
    else:
        matcher = build_vectorized_matcher(
            processed_train_strings,
            matcher_type=spec["matcher_type"],
            ngram_range=spec["ngram_range"],
        )

    database_build_ms = (time.perf_counter() - start_build) * 1000
    shortened_characters = original_key_length - processed_key_length
    shortened_fraction = shortened_characters / original_key_length if original_key_length else 0.0

    return {
        "spec": spec,
        "matcher": matcher,
        "database_build_ms": database_build_ms,
        "original_key_length": original_key_length,
        "processed_key_length": processed_key_length,
        "shortened_characters": shortened_characters,
        "shortened_fraction": shortened_fraction,
    }

In [62]:
def predict_cached_query_for_approach(example, matcher_bundle):
    """Predict one cached query string with one approach and split timing by stage.

    Args:
        example: Cached query dictionary built from `cached_midi_file_to_note_strings`.
        matcher_bundle: Dictionary returned by `build_approach_matcher`.

    Returns:
        A dictionary containing correctness plus extraction, compression, matching,
        and combined read-to-prediction time for one query.
    """
    import time

    spec = matcher_bundle["spec"]

    start_compression = time.perf_counter()
    processed_query = spec["transform"](example["base_string"])
    compression_ms = (time.perf_counter() - start_compression) * 1000

    start_match = time.perf_counter()
    if spec["matcher_type"] == "indexed_jaccard":
        prediction = match_song_with_index(processed_query, matcher_bundle["matcher"])
    else:
        prediction = match_with_vectorized_matcher(processed_query, matcher_bundle["matcher"])
    matching_ms = (time.perf_counter() - start_match) * 1000

    query_ms = example["extraction_ms"] + compression_ms + matching_ms
    return {
        "correct": prediction["predicted_song"] == example["true_song"],
        "extraction_ms": example["extraction_ms"],
        "compression_ms": compression_ms,
        "matching_ms": matching_ms,
        "query_ms": query_ms,
    }

## Scoring helpers

These functions turn benchmark counts into reportable metrics.


In [63]:
def accuracy(correct_count, total_count):
    """Convert a correct-count and total-count pair to an accuracy value.

    Args:
        correct_count: Number of correct predictions.
        total_count: Number of evaluated files.

    Returns:
        Accuracy as a float from 0.0 to 1.0, or 0.0 when there are no evaluated files.
    """
    return correct_count / total_count if total_count else 0.0


## Unified approach catalog

Each approach below is a combination of three choices:

1. **Note selection**: how notes are extracted from simultaneous MIDI onsets.
2. **String postprocessing**: how the extracted base string is compressed or abstracted.
3. **Matching algorithm**: how each processed query string is compared with the training database.


In [64]:
EXTRACTION_METHODS = {
    "highest": {
        "label": "Highest notes",
        "description": "Select the highest note in each 80 ms onset group, treating it as a melody proxy.",
    },
    "lowest": {
        "label": "Lowest notes",
        "description": "Select the lowest note in each 80 ms onset group, approximating the bass line.",
    },
    "top2": {
        "label": "Top-2 notes",
        "description": "Keep up to the two highest notes from each onset group for a compact chord/melody snapshot.",
    },
    "top3": {
        "label": "Top-3 notes",
        "description": "Keep up to the three highest notes from each onset group for a denser chord snapshot.",
    },
}

COMPRESSION_METHODS = {
    "identity": {
        "label": "Raw string",
        "description": "Use the extracted string without postprocessing.",
        "transform": lambda s: s,
    },
    "motifs_modal8": {
        "label": "Motif collapse + modal 8-blocks",
        "description": "Collapse repeated adjacent motifs of length 2-8, then summarize each 8-character block by its modal pitch character.",
        "transform": lambda s: modal_block_signature(collapse_repeated_motifs(s), block_size=8),
    },
    "modal16": {
        "label": "Modal 16-blocks",
        "description": "Split the string into 16-character blocks and replace each block with its most common pitch character.",
        "transform": lambda s: modal_block_signature(s, block_size=16),
    },
    "motifs": {
        "label": "Motif collapse",
        "description": "Collapse adjacent repeated motifs of length 2-8 while keeping the remaining pitch sequence intact.",
        "transform": lambda s: collapse_repeated_motifs(s),
    },
    "pitch_class": {
        "label": "Pitch class",
        "description": "Convert absolute pitches to 12 pitch classes and collapse adjacent repeated pitch classes.",
        "transform": lambda s: pitch_class_signature(s),
    },
    "first4": {
        "label": "First-in-4 blocks",
        "description": "Downsample by keeping the first character from every 4-character block.",
        "transform": lambda s: first_character_blocks(s, block_size=4),
    },
    "repeated_run_min3": {
        "label": "Repeated-run min3",
        "description": "Keep notes only when they appear in adjacent runs of at least three repeats, then collapse each run.",
        "transform": lambda s: repeated_run_signature(s, min_run_length=3),
    },
    "pitch_band8": {
        "label": "Eight pitch bands",
        "description": "Map notes into eight broad pitch-height bands and collapse adjacent repeated bands.",
        "transform": lambda s: pitch_band_signature(s, band_count=8),
    },
    "contour": {
        "label": "Contour",
        "description": "Convert adjacent notes into up/down/same movement symbols and collapse repeated movement directions.",
        "transform": lambda s: contour_signature(s),
    },
    "zlib_base85": {
        "label": "zlib/base85",
        "description": "Compress the string with zlib and encode the bytes as base85 text.",
        "transform": lambda s: zlib_base85_encode(s),
    },
}

MATCHER_METHODS = {
    "tfidf_1_1": {
        "label": "TF-IDF char 1",
        "description": "TF-IDF character unigram cosine similarity.",
        "matcher_type": "tfidf",
        "ngram_range": (1, 1),
    },
    "tfidf_2_2": {
        "label": "TF-IDF char 2",
        "description": "TF-IDF character bigram cosine similarity.",
        "matcher_type": "tfidf",
        "ngram_range": (2, 2),
    },
    "tfidf_3_3": {
        "label": "TF-IDF char 3",
        "description": "TF-IDF character trigram cosine similarity.",
        "matcher_type": "tfidf",
        "ngram_range": (3, 3),
    },
    "tfidf_2_4": {
        "label": "TF-IDF char 2-4",
        "description": "TF-IDF character n-gram cosine similarity with 2-, 3-, and 4-grams.",
        "matcher_type": "tfidf",
        "ngram_range": (2, 4),
    },
    "tfidf_4_4": {
        "label": "TF-IDF char 4",
        "description": "TF-IDF character 4-gram cosine similarity.",
        "matcher_type": "tfidf",
        "ngram_range": (4, 4),
    },
    "tfidf_3_5": {
        "label": "TF-IDF char 3-5",
        "description": "TF-IDF character n-gram cosine similarity with 3-, 4-, and 5-grams.",
        "matcher_type": "tfidf",
        "ngram_range": (3, 5),
    },
    "binary_2_2": {
        "label": "Binary char 2",
        "description": "Binary character bigram cosine similarity.",
        "matcher_type": "binary",
        "ngram_range": (2, 2),
    },
    "binary_3_5": {
        "label": "Binary char 3-5",
        "description": "Binary character n-gram cosine similarity with 3-, 4-, and 5-grams.",
        "matcher_type": "binary",
        "ngram_range": (3, 5),
    },
    "jaccard_3": {
        "label": "Indexed Jaccard char 3",
        "description": "Exact lookup first, then inverted-index Jaccard similarity on character trigrams.",
        "matcher_type": "indexed_jaccard",
        "ngram_range": (3, 3),
    },
}

APPROACH_BLUEPRINTS = [
    ("highest", "identity", "tfidf_1_1"),
    ("highest", "identity", "tfidf_3_3"),
    ("highest", "identity", "jaccard_3"),
    ("highest", "identity", "binary_3_5"),
    ("lowest", "identity", "tfidf_3_3"),
    ("top2", "identity", "tfidf_3_3"),
    ("top3", "identity", "tfidf_3_3"),
    ("highest", "motifs_modal8", "tfidf_1_1"),
    ("top2", "modal16", "tfidf_2_2"),
    ("top2", "motifs", "tfidf_3_3"),
    ("highest", "motifs_modal8", "tfidf_2_2"),
    ("top2", "pitch_class", "tfidf_2_4"),
    ("top2", "motifs_modal8", "tfidf_1_1"),
    ("top2", "motifs_modal8", "tfidf_2_2"),
    ("highest", "first4", "tfidf_2_4"),
    ("top2", "motifs", "tfidf_4_4"),
    ("top2", "motifs", "tfidf_3_5"),
    ("highest", "first4", "tfidf_3_3"),
    ("top2", "pitch_class", "tfidf_2_2"),
    ("top2", "pitch_class", "tfidf_3_3"),
    ("highest", "pitch_class", "tfidf_1_1"),
    ("lowest", "pitch_class", "tfidf_4_4"),
    ("lowest", "identity", "binary_3_5"),
    ("top2", "motifs", "tfidf_2_4"),
    ("top2", "identity", "tfidf_3_5"),
    ("top2", "identity", "tfidf_3_3"),
    ("top2", "modal16", "tfidf_3_3"),
    ("highest", "repeated_run_min3", "jaccard_3"),
    ("highest", "pitch_band8", "jaccard_3"),
    ("highest", "contour", "jaccard_3"),
    ("top3", "motifs_modal8", "tfidf_1_1"),
    ("highest", "zlib_base85", "binary_2_2"),
]


def build_approach_spec(index, extraction_key, compression_key, matcher_key):
    """Build one reusable approach specification from method keys.

    Args:
        index: One-based approach index for display.
        extraction_key: Key in `EXTRACTION_METHODS`.
        compression_key: Key in `COMPRESSION_METHODS`.
        matcher_key: Key in `MATCHER_METHODS`.

    Returns:
        An approach specification dictionary used by the benchmark.
    """
    extraction = EXTRACTION_METHODS[extraction_key]
    compression = COMPRESSION_METHODS[compression_key]
    matcher = MATCHER_METHODS[matcher_key]
    return {
        "id": f"{index:02d}",
        "name": f"{extraction['label']} + {compression['label']} + {matcher['label']}",
        "extraction": extraction_key,
        "extraction_label": extraction["label"],
        "note_selection": extraction["description"],
        "compression": compression_key,
        "compression_label": compression["label"],
        "processing": compression["description"],
        "transform": compression["transform"],
        "matcher": matcher_key,
        "matcher_label": matcher["label"],
        "matching": matcher["description"],
        "matcher_type": matcher["matcher_type"],
        "ngram_range": matcher["ngram_range"],
    }


APPROACH_SPECS = [build_approach_spec(index, extraction_key, compression_key, matcher_key) for index, (extraction_key, compression_key, matcher_key) in enumerate(APPROACH_BLUEPRINTS, start=1)]

if SELECTED_APPROACH_ID is None:
    ACTIVE_APPROACH_SPECS = APPROACH_SPECS
else:
    ACTIVE_APPROACH_SPECS = [spec for spec in APPROACH_SPECS if spec["id"] == SELECTED_APPROACH_ID]
    if not ACTIVE_APPROACH_SPECS:
        raise ValueError(f"No approach found for SELECTED_APPROACH_ID={SELECTED_APPROACH_ID!r}")

approach_catalog_df = pd.DataFrame(
    [
        {
            "Approach": spec["id"],
            "Name": spec["name"],
            "Note selection": spec["note_selection"],
            "String compression": spec["processing"],
            "Matching algorithm": spec["matching"],
        }
        for spec in ACTIVE_APPROACH_SPECS
    ]
)

approach_catalog_df.style.set_caption("Active approach catalog")

,Approach,Name,Note selection,String compression,Matching algorithm
0,01,Highest notes + Raw string + TF-IDF char 1,"Select the highest note in each 80 ms onset group, treating it as a melody proxy.",Use the extracted string without postprocessing.,TF-IDF character unigram cosine similarity.
1,02,Highest notes + Raw string + TF-IDF char 3,"Select the highest note in each 80 ms onset group, treating it as a melody proxy.",Use the extracted string without postprocessing.,TF-IDF character trigram cosine similarity.
2,03,Highest notes + Raw string + Indexed Jaccard char 3,"Select the highest note in each 80 ms onset group, treating it as a melody proxy.",Use the extracted string without postprocessing.,"Exact lookup first, then inverted-index Jaccard similarity on character trigrams."
3,04,Highest notes + Raw string + Binary char 3-5,"Select the highest note in each 80 ms onset group, treating it as a melody proxy.",Use the extracted string without postprocessing.,"Binary character n-gram cosine similarity with 3-, 4-, and 5-grams."
4,05,Lowest notes + Raw string + TF-IDF char 3,"Select the lowest note in each 80 ms onset group, approximating the bass line.",Use the extracted string without postprocessing.,TF-IDF character trigram cosine similarity.
5,06,Top-2 notes + Raw string + TF-IDF char 3,Keep up to the two highest notes from each onset group for a compact chord/melody snapshot.,Use the extracted string without postprocessing.,TF-IDF character trigram cosine similarity.
6,07,Top-3 notes + Raw string + TF-IDF char 3,Keep up to the three highest notes from each onset group for a denser chord snapshot.,Use the extracted string without postprocessing.,TF-IDF character trigram cosine similarity.
7,08,Highest notes + Motif collapse + modal 8-blocks + TF-IDF char 1,"Select the highest note in each 80 ms onset group, treating it as a melody proxy.","Collapse repeated adjacent motifs of length 2-8, then summarize each 8-character block by its modal pitch character.",TF-IDF character unigram cosine similarity.
8,09,Top-2 notes + Modal 16-blocks + TF-IDF char 2,Keep up to the two highest notes from each onset group for a compact chord/melody snapshot.,Split the string into 16-character blocks and replace each block with its most common pitch character.,TF-IDF character bigram cosine similarity.
9,10,Top-2 notes + Motif collapse + TF-IDF char 3,Keep up to the two highest notes from each onset group for a compact chord/melody snapshot.,Collapse adjacent repeated motifs of length 2-8 while keeping the remaining pitch sequence intact.,TF-IDF character trigram cosine similarity.


## Select dataset and cache extracted strings

Use `DATASET_NAME` near the top of the notebook to switch among `mini`, `maestro`, and `maestro_multi`. Extracted strings are persisted in `EXTRACTION_CACHE_PATH` with file size and modification time keys, so repeated notebook runs avoid reparsing unchanged MIDI files. Query files are still represented with extraction timing, but cache hits measure cache retrieval rather than MIDI parsing.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

note_character_map = build_piano_note_character_map()
extraction_cache = load_extraction_cache(EXTRACTION_CACHE_PATH, enabled=USE_EXTRACTION_CACHE)

all_song_dirs = sorted(path for path in SONGS_DIR.iterdir() if path.is_dir())
if REQUIRE_MULTIPLE_RECORDINGS:
    eligible_song_names = {path.name for path in all_song_dirs if len(list(path.glob("*.mid*"))) > 1}
else:
    eligible_song_names = {path.name for path in all_song_dirs}

train_midi_files = sorted(path for path in TRAIN_DIR.glob("*.mid*") if path.stem in eligible_song_names)
evaluation_examples = [{"path": path, "true_song": path.parent.name} for path in sorted(SONGS_DIR.rglob("*.mid*")) if path.parent.name in eligible_song_names]
required_extractions = sorted({spec["extraction"] for spec in ACTIVE_APPROACH_SPECS})

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    train_records = list(
        executor.map(
            lambda path: (
                path.stem,
                *cached_midi_file_to_note_strings(
                    path,
                    note_character_map,
                    required_extractions,
                    extraction_cache,
                ),
            ),
            train_midi_files,
        )
    )

for _, _, _, cache_key, updated_entry in train_records:
    extraction_cache[cache_key] = updated_entry

train_strings_by_extraction = {extraction: {song_name: strings[extraction] for song_name, strings, _, _, _ in train_records} for extraction in required_extractions}

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    query_records = list(
        executor.map(
            lambda example: (
                example,
                *cached_midi_file_to_note_strings(
                    example["path"],
                    note_character_map,
                    required_extractions,
                    extraction_cache,
                ),
            ),
            evaluation_examples,
        )
    )

for _, _, _, cache_key, updated_entry in query_records:
    extraction_cache[cache_key] = updated_entry

cached_query_records = [
    {
        extraction: {
            "path": example["path"],
            "true_song": example["true_song"],
            "base_string": strings[extraction],
            "extraction_ms": extraction_ms,
        }
        for extraction in required_extractions
    }
    for example, strings, extraction_ms, _, _ in query_records
]

save_extraction_cache(EXTRACTION_CACHE_PATH, extraction_cache, enabled=USE_EXTRACTION_CACHE)

evaluation_strings_by_extraction = {extraction: [record[extraction] for record in cached_query_records] for extraction in required_extractions}

extraction_summary_df = (
    pd.DataFrame(
        [
            {
                "Dataset": DATASET_NAME,
                "Extraction": extraction,
                "Training songs": len(train_strings),
                "Evaluation files": len(evaluation_strings_by_extraction[extraction]),
                "Eligible songs": len(eligible_song_names),
                "Total training characters": sum(len(value) for value in train_strings.values()),
                "Mean training characters": round(sum(len(value) for value in train_strings.values()) / len(train_strings), 1),
                "Mean query extraction ms": sum(example["extraction_ms"] for example in evaluation_strings_by_extraction[extraction]) / len(evaluation_strings_by_extraction[extraction]),
            }
            for extraction, train_strings in train_strings_by_extraction.items()
        ]
    )
    .sort_values("Extraction")
    .reset_index(drop=True)
)

extraction_summary_df.style.format({"Mean query extraction ms": "{:.3f}"}).set_caption("Dataset and extracted-string summary")

## Benchmark unified approaches

Approach databases are built in parallel. Query matching also runs in parallel against cached extracted strings. The reported `Avg query ms` is the average cached MIDI extraction time plus compression time plus matching time for a single query; database creation is reported separately and is not included in that query metric.


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

start_all_builds = time.perf_counter()
with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    matcher_bundles = list(
        executor.map(
            lambda spec: build_approach_matcher(
                spec,
                train_strings_by_extraction[spec["extraction"]],
            ),
            ACTIVE_APPROACH_SPECS,
        )
    )
parallel_database_build_wall_ms = (time.perf_counter() - start_all_builds) * 1000

unified_result_rows = []

for matcher_bundle in matcher_bundles:
    spec = matcher_bundle["spec"]
    cached_examples = evaluation_strings_by_extraction[spec["extraction"]]

    with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
        query_results = list(
            executor.map(
                lambda example: predict_cached_query_for_approach(example, matcher_bundle),
                cached_examples,
            )
        )

    correct_count = sum(result["correct"] for result in query_results)
    total_count = len(query_results)
    total_extraction_ms = sum(result["extraction_ms"] for result in query_results)
    total_compression_ms = sum(result["compression_ms"] for result in query_results)
    total_matching_ms = sum(result["matching_ms"] for result in query_results)
    total_query_ms = sum(result["query_ms"] for result in query_results)
    mean_extraction_ms = total_extraction_ms / total_count if total_count else 0.0
    mean_compression_ms = total_compression_ms / total_count if total_count else 0.0
    mean_matching_ms = total_matching_ms / total_count if total_count else 0.0
    mean_query_ms = total_query_ms / total_count if total_count else 0.0
    approach_accuracy = accuracy(correct_count, total_count)
    accuracy_time_score = (approach_accuracy**4) * 1000 / mean_query_ms if mean_query_ms else 0.0

    unified_result_rows.append(
        {
            "Approach": spec["id"],
            "Name": spec["name"],
            "Extraction": spec["extraction_label"],
            "String compression": spec["compression_label"],
            "Matcher": spec["matcher_label"],
            "Accuracy": approach_accuracy,
            "Accuracy-time score": accuracy_time_score,
            "Avg query ms": mean_query_ms,
            "Avg extraction ms": mean_extraction_ms,
            "Avg compression ms": mean_compression_ms,
            "Avg matching ms": mean_matching_ms,
            "Database build ms": matcher_bundle["database_build_ms"],
            "Database key length": matcher_bundle["processed_key_length"],
            "Shortened %": matcher_bundle["shortened_fraction"],
        }
    )

result_column_order = [
    "Rank",
    "Approach",
    "Name",
    "Extraction",
    "String compression",
    "Matcher",
    "Accuracy",
    "Accuracy-time score",
    "Avg query ms",
    "Avg extraction ms",
    "Avg compression ms",
    "Avg matching ms",
    "Shortened %",
    "Database key length",
    "Database build ms",
]

unified_results_df = pd.DataFrame(unified_result_rows).sort_values(["Accuracy-time score", "Accuracy", "Avg query ms"], ascending=[False, False, True]).reset_index(drop=True)
unified_results_df.insert(0, "Rank", range(1, len(unified_results_df) + 1))
unified_results_df = unified_results_df[result_column_order]

## Unified results

This single table is the final report. The columns are ordered as approach identity, approach components, accuracy/score, timing breakdown, and size metrics. The table is sorted by the accuracy-time score, which estimates speed-adjusted quality as `accuracy ** 4 * 1000 / avg_query_ms` so lower-accuracy approaches are penalized much more heavily than small runtime differences.


In [ ]:
display(
    unified_results_df.style.format(
        {
            "Accuracy": "{:.1%}",
            "Accuracy-time score": "{:.3f}",
            "Avg query ms": "{:.3f}",
            "Avg extraction ms": "{:.3f}",
            "Avg compression ms": "{:.3f}",
            "Avg matching ms": "{:.3f}",
            "Database build ms": "{:.2f}",
            "Database key length": "{:,}",
            "Shortened %": "{:.1%}",
        }
    )
    .set_caption("Unified benchmark results")
    .hide(axis="index")
)

Rank,Approach,Name,Extraction,String compression,Matcher,Accuracy,Accuracy-time score,Avg query ms,Avg extraction ms,Avg compression ms,Avg matching ms,Shortened %,Database key length,Database build ms
1,28,Highest notes + Repeated-run min3 + Indexed Jaccard char 3,Highest notes,Repeated-run min3,Indexed Jaccard char 3,100.0%,14661.894,0.068,0.000,0.062,0.006,98.4%,232,0.29
2,03,Highest notes + Raw string + Indexed Jaccard char 3,Highest notes,Raw string,Indexed Jaccard char 3,100.0%,2866.601,0.349,0.000,0.000,0.348,0.0%,"14,862",4.15
3,21,Highest notes + Pitch class + TF-IDF char 1,Highest notes,Pitch class,TF-IDF char 1,100.0%,2362.928,0.423,0.000,0.191,0.232,18.7%,"12,082",2.31
4,29,Highest notes + Eight pitch bands + Indexed Jaccard char 3,Highest notes,Eight pitch bands,Indexed Jaccard char 3,91.3%,2285.397,0.304,0.000,0.229,0.075,48.8%,"7,603",1.57
5,27,Top-2 notes + Modal 16-blocks + TF-IDF char 3,Top-2 notes,Modal 16-blocks,TF-IDF char 3,100.0%,1428.368,0.700,0.000,0.342,0.358,93.7%,"1,486",106.77
6,08,Highest notes + Motif collapse + modal 8-blocks + TF-IDF char 1,Highest notes,Motif collapse + modal 8-blocks,TF-IDF char 1,100.0%,1159.893,0.862,0.000,0.703,0.159,88.1%,"1,770",542.98
7,19,Top-2 notes + Pitch class + TF-IDF char 2,Top-2 notes,Pitch class,TF-IDF char 2,100.0%,1003.349,0.997,0.000,0.306,0.690,19.4%,"19,152",10.03
8,13,Top-2 notes + Motif collapse + modal 8-blocks + TF-IDF char 1,Top-2 notes,Motif collapse + modal 8-blocks,TF-IDF char 1,100.0%,762.664,1.311,0.000,1.137,0.174,88.1%,"2,835",552.99
9,31,Top-3 notes + Motif collapse + modal 8-blocks + TF-IDF char 1,Top-3 notes,Motif collapse + modal 8-blocks,TF-IDF char 1,100.0%,692.138,1.445,0.000,1.272,0.173,88.0%,"3,384",15.32
10,09,Top-2 notes + Modal 16-blocks + TF-IDF char 2,Top-2 notes,Modal 16-blocks,TF-IDF char 2,100.0%,540.337,1.851,0.000,0.353,1.497,93.7%,"1,486",603.62


## Best approach

The best approach is the first row of the unified table: the highest accuracy-weighted speed score, with accuracy and query time as tie-breakers.


In [ ]:
best_attempt = unified_results_df.iloc[0]
print(
    f"Best approach by accuracy-time score: {best_attempt['Approach']} {best_attempt['Name']} "
    f"with score {best_attempt['Accuracy-time score']:.3f}, "
    f"{best_attempt['Accuracy']:.1%} accuracy, "
    f"{best_attempt['Avg query ms']:.3f} ms avg query time, and "
    f"{best_attempt['Shortened %']:.1%} string shortening."
)

Best approach by accuracy-time score: 28 Highest notes + Repeated-run min3 + Indexed Jaccard char 3 with score 14661.894, 100.0% accuracy, 0.068 ms avg query time, and 98.4% string shortening.
